## Pydantic 模型

它通过在运行时强制执行类型提示，确保数据的正确性和一致性，是生产场景首选。

需要满足的几个要素：
- 所有结构化输出的数据模型都必须继承BaseModel
- 使用类型提示。Pydantic 支持丰富的字段类型：str、int、float、List[xxx]、Optional[xxx]等
- 使用Field()添加字段默认值和描述，帮助LLM 理解字段含义


（1）使用with_structured_output 即可引导模型进行结构化输出

举例1：直接断句那种类型

In [2]:
from typing import Optional, Literal

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

#优先加载配置文件
load_dotenv(override=True)

#初始化大模型
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

from pydantic import BaseModel, Field, ValidationError


#顶一个Pydantic类型
class Person(BaseModel):
    """
    人物信息
    """
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的LLM
structured_llm = model_openai.with_structured_output(Person)

#调用
result = structured_llm.invoke("张三是一名30岁的软件工程师")

print(result)
print(type(result))

# result是Person实例
print(result.name)
print(result.age)
print(result.occupation)


name='张三' age=30 occupation='软件工程师'
<class '__main__.Person'>
张三
30
软件工程师


举例2：从互联网上搜索和整理

In [3]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

class MovieModel(BaseModel):
    """
    电影的详细信息
    """
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")

model_with_structure = model_openai.with_structured_output(MovieModel)
response = model_with_structure.invoke("给出盗梦空间的信息")
print(response)
print(type(response))


title='盗梦空间' year=2010 director='克里斯托弗·诺兰' rating=9.4
<class '__main__.MovieModel'>


举例3：文字的理解和分析推理

In [4]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

class SentimentAnalysis(BaseModel):
    """
    情感分析结果
    """
    sentiment: str = Field(description="情感倾向：positive/negative/neutral")
    confidence: float = Field(description="置信度，0-1之间")
    keywords: list[str] = Field(description="关键词列表")

model_with_structure = model_openai.with_structured_output(SentimentAnalysis)

text = "这个课程内容很实用，学到了很多知识，强烈推荐！"

response = model_with_structure.invoke(f"分析以下情感：{text}")

print(response)
print(type(response))

sentiment='positive' confidence=0.98 keywords=['实用', '学到了很多知识', '强烈推荐']
<class '__main__.SentimentAnalysis'>


情况1：可选字段

如果LLM中未填充某些字段，就使用Optional指定字段为可选的。

In [13]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Optional
import os

#优先加载配置文件
load_dotenv(override=True)

#初始化大模型
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

#顶一个Pydantic类型
class Person(BaseModel):
    """
    人物信息
    """
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的LLM
structured_llm = model_openai.with_structured_output(Person)

#调用,如果用户消息里，没有年龄的化，就会忽略掉
result = structured_llm.invoke("张三是一名软件工程师")

print(result)


name='张三' age=None occupation='软件工程师'


情况2：默认值（注意：不同模型提供商对default字段的支持是不同的。）

In [16]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Optional
import os

#优先加载配置文件
load_dotenv(override=True)

#初始化大模型
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

#顶一个Pydantic类型
class Product(BaseModel):
    """产品信息"""
    name: str = Field(description="产品名称")
    price: float = Field(description="价格")
    description: Optional[str] = Field(description="产品描述")
    stock: int = Field(default=100, description="库存")

structured_llm = model_openai.with_structured_output(Product)
print("\n场景1：完整信息")
result1 = structured_llm.invoke("iPhone 15 售价5999 元，最新款智能手机，库存50台")
print(result1)
print("\n场景2：缺少描述和库存")
result2 = structured_llm.invoke("MacBook Pro 售价12999 元")
print(result2)



场景1：完整信息
name='iPhone 15' price=5999.0 description='最新款智能手机' stock=50

场景2：缺少描述和库存
name='search' price=12999.0 description='MacBook Pro 售价信息已获取，当前价格为12999元，符合市场常规定价区间。建议关注苹果官方渠道或授权经销商以确认具体型号和配置版本（如M2/M3芯片、内存/存储组合等），因不同配置价格差异较大。此报价可能包含促销优惠，实际支付时请以结账页面为准。如需购买，可访问 apple.com 或京东/天猫官方旗舰店比对最新活动价。' stock=0


情况3：枚举类型

In [19]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum

load_dotenv(override=True)
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

#定义你的优先级枚举类
class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"

class CustomerInfo(BaseModel):
    """
    客户消息
    """
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")

structured_llm = model_openai.with_structured_output(CustomerInfo)

conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""
result = structured_llm .invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)

print("\n提取结果：")
print(f"  客户: {result.name}")
print(f"  电话: {result.phone}")
print(f"  邮箱: {result.email or '未提供'}")
print(f"  问题: {result.issue}")
print(f"  紧急程度: {result.urgency.value}")


name='王小明' phone='138-1234-5678' email=None issue='订单未发货' urgency=<Priority.HIGH: '高'>

提取结果：
  客户: 王小明
  电话: 138-1234-5678
  邮箱: 未提供
  问题: 订单未发货
  紧急程度: 高


In [21]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Optional, Literal

load_dotenv(override=True)
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

#采用Literal
class CustomerInfo(BaseModel):
    """
    客户消息
    """
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    # 使用Literal 直接限定字面量值
    urgency: Literal["低", "中", "高"] = Field(description="紧急程度")

structured_llm = model_openai.with_structured_output(CustomerInfo)

conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""
result = structured_llm .invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)

print("\n提取结果：")
print(f"  客户: {result.name}")
print(f"  电话: {result.phone}")
print(f"  邮箱: {result.email or '未提供'}")
print(f"  问题: {result.issue}")
#不用enum了。不需要.value
print(f"  紧急程度: {result.urgency}")

name='王小明' phone='138-1234-5678' email=None issue='订单未发货' urgency='高'

提取结果：
  客户: 王小明
  电话: 138-1234-5678
  邮箱: 未提供
  问题: 订单未发货
  紧急程度: 高


情况4：列表提取

In [22]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List

load_dotenv(override=True)
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

class Person(BaseModel):
    """人物信息"""
    name : str
    age : int

class PersonList(BaseModel):
    """人物列表信息"""
    people: List[Person]  # 多个Person 对象

structured_llm = model_openai.with_structured_output(PersonList)

result = structured_llm .invoke("张三 30岁，李四 25岁")
print(result)


people=[Person(name='张三', age=30), Person(name='李四', age=25)]


In [23]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List

load_dotenv(override=True)
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

class Review(BaseModel):
    """产品评论"""
    product : str
    rating: int = Field(description="评分1-5")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")

structured_llm = model_openai.with_structured_output(Review)

result = structured_llm .invoke("""
 iPhone 17 很棒！摄像头强大，手感好。但是价格贵，没有充电器。4分。
""")

print(result)

product='iPhone 17' rating=4 pros=['摄像头强大', '手感好'] cons=['价格贵', '没有充电器']


情况5：嵌套结构 (LLM 能力有限，复杂嵌套结构可能会出错。所以建议：嵌套层级 ≤ 3 层)

In [25]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List

load_dotenv(override=True)
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

class Aspect(BaseModel):
    """评论维度"""
    name: str = Field(description="维度名称，如：质量、价格、服务")
    score: int = Field(description="评分，1-5")
    comment: str = Field(description="具体评价")

class ProductReview(BaseModel):
    """产品评论分析"""
    overall_sentiment: str = Field(description="整体情感：positive/negative/neutral")
    overall_score: int = Field(description="综合评分，1-5")
    aspects: List[Aspect] = Field(description="各维度评价")
    summary: str = Field(description="一句话总结")

structured_llm = model_openai.with_structured_output(ProductReview)
# 测试
review_text = """
这款笔记本电脑性能非常强大，运行大型软件毫无压力。
屏幕色彩鲜艳，看视频很舒服。
不过价格有点贵，而且风扇噪音较大。
客服态度很好，物流也快。
总体来说还是值得购买的。
"""
result = structured_llm .invoke(f"分析以下产品评论：\n{review_text}")

print(f"整体情感: {result.overall_sentiment}")
print(f"综合评分: {result.overall_score}/5")
print(f"\n各维度评价:")
for aspect in result.aspects:
    print(f" - {aspect.name}: {aspect.score}/5 - {aspect.comment}")

print(f"\n总结: {result.summary}")

整体情感: positive
综合评分: 4/5

各维度评价:
 - performance: 5/5 - 性能非常强大，运行大型软件毫无压力。
 - display: 4/5 - 屏幕色彩鲜艳，看视频很舒服。
 - price: -1/5 - 价格有点贵。
 - cooling_system: -2/5 - 风扇噪音较大。
 - customer_service: 5/5 - 客服态度很好。
 - logistics: 5/5 - 物流也快。

总结: 该评论总体情感为正面（4/5）。用户高度认可产品的性能、屏幕显示效果以及配套的客户服务和物流体验。然而，用户对价格和散热系统的噪音表现提出了负面反馈。尽管存在这些缺点，用户最终仍认为该产品值得购买。


情况6：限制条件

In [31]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError

load_dotenv(override=True)
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

class User(BaseModel):
    name: str = Field(min_length=2, max_length=10)
    age: int = Field(ge=0, le=150)
    email: str

print("有效数据：")
try:
    user = User(name="张三", age=30, email="zhangsan@hotmail.com")
    print(f"[OK] {user.name}, {user.age}, {user.email}")
except ValidationError as e:
    print(f"[FAIL] {e}")

print("\n无效数据（年龄超出范围，名字也超出范围）")
try:
    user = User(name="张三这个人大家如何看到的", age=300, email="zhangsan@hotmail.com")
    print(f"[OK] {user.name}, {user.age}, {user.email}")
except ValidationError as e:
    print(f"[FAIL] 验证失败（符合预期）：{e.errors()[0]['msg']}")


有效数据：
[OK] 张三, 30, zhangsan@hotmail.com

无效数据（年龄超出范围，名字也超出范围）
[FAIL] 验证失败（符合预期）：String should have at most 10 characters
